# SAVE v0.2.0 — Python API & CLI Quickstart Tutorial

This self-contained tutorial walks through the unified **`sc-save`** (`sc_save`) Python API and CLI:
1. **Gene Block Construction** (`build_gene_blocks`): partition genes into balanced pathway-coherent blocks via Optimal Transport.
2. **Model Configuration & Training** (`SAVEConfig`, `SAVE.fit`): train the Stage-1 Attention VAE and Stage-2 Latent Flow Matching model.
3. **Representation & Decoding** (`encode`, `decode`, `reconstruct`).
4. **Conditional Generation** (`generate`, single-pass batched `generate_like`, and `generate_with_attention`).
5. **Zero-Shot Condition / Batch Transfer** (`transfer`).
6. **Save & Reload Bundles** (`SAVE.save`, `SAVE.load`).
7. **Unified `sc-save` CLI** (`train`, `generate`, `bec`, `perturb`, `evaluate`, `reproduce`).

For full ICLR 2026 benchmark reproduction (Tables 1–5), see [`SAVE_ICLR2026_Reproduce_Colab.ipynb`](SAVE_ICLR2026_Reproduce_Colab.ipynb) and [`REPRODUCIBILITY.md`](../REPRODUCIBILITY.md).

## 1. Installation

Install `sc-save` from PyPI (`pip install sc-save`) or editable from a local repository clone without replacing preinstalled CUDA PyTorch.

In [ ]:
import importlib.util
from pathlib import Path
import subprocess
import sys

if importlib.util.find_spec("sc_save") is None:
    repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if (repo_root / "pyproject.toml").exists() and (repo_root / "src" / "sc_save").exists():
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation", "-e", str(repo_root)],
            check=True,
        )
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sc-save"], check=True)

import sc_save
import torch
print(f"sc_save version: {sc_save.__version__} | PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

## 2. Build Semantic Gene Blocks (`build_gene_blocks`)

In real experiments, load your preprocessed `.h5ad` and precomputed GenePT embeddings (`load_gene_embeddings("GenePT_gene_protein_embedding_model_3_text.pickle")`). Below we generate a compact synthetic multi-batch, multi-cell-type scRNA-seq dataset so the entire notebook runs end-to-end in seconds.

In [ ]:
import tempfile
from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

from sc_save import build_gene_blocks

rng = np.random.default_rng(42)
n_cells = 120
n_genes = 60
genes = [f"GENE_{i}" for i in range(50)] + [f"EXTRA_{i}" for i in range(10)]

batches = ["10x_v2", "10x_v3"] * (n_cells // 2)
cell_types = (["CD4T", "CD4T", "B", "B", "Mono", "Mono"] * (n_cells // 6))[:n_cells]
counts = rng.poisson(lam=2.5, size=(n_cells, n_genes)).astype(np.float32)
# Inject mild cell-type signal
for i, ct in enumerate(cell_types):
    if ct == "CD4T":
        counts[i, :10] += 3.0
    elif ct == "B":
        counts[i, 10:20] += 3.0
    else:
        counts[i, 20:30] += 3.0

adata_raw = ad.AnnData(
    X=sparse.csr_matrix(np.log1p(counts)),
    obs=pd.DataFrame(
        {"batch": batches, "cell_type": cell_types},
        index=[f"cell_{i}" for i in range(n_cells)],
    ),
    var=pd.DataFrame(index=genes),
)

# Synthetic 16-dim semantic embeddings for the 50 covered genes
gene_embeddings = {f"GENE_{i}": rng.normal(size=16).astype(np.float32) for i in range(50)}

# Partition into balanced semantic gene blocks of size 20 (40 clustered + 20 remainder = 60)
block_res = build_gene_blocks(adata_raw, gene_embeddings, block_size=20, ot_max_iter=10)
print(
    f"GeneBlockResult: shape={block_res.adata.shape}, "
    f"shared={block_res.n_shared_genept}, "
    f"blocks={block_res.n_semantic_blocks}, "
    f"pad={block_res.n_pad}, "
    f"block_sizes={block_res.block_sizes}"
)
block_res.adata.var.head()

## 3. Configure & Train `SAVE` (`SAVEConfig` + `SAVE.fit`)

- For full benchmarks (e.g. `block_size=3200`), use `SAVEConfig.save_b(gene_block_size=3200)` or `SAVEConfig.save_b_perturb(gene_block_size=400)`.
- Here we use `SAVEConfig.smoke(gene_block_size=20)` with `cond_encoding="offset"` (collision-free condition IDs across columns, reserving index `0` exclusively for `<mask>`).

In [ ]:
from sc_save import SAVE, SAVEConfig

cfg = SAVEConfig.smoke(
    gene_block_size=20,
    latent_block_size=20,
    num_heads=2,
    vae_epochs=3,
    fm_epochs=3,
    batch_size=32,
    sample_steps=10,
    cond_encoding="offset",
)

model = SAVE(
    block_res.adata,
    condition_cols=("batch", "cell_type"),
    config=cfg,
    device="auto",
    cond_encoding="offset",
)
model.fit()

## 4. Encode, Reconstruct, Generate & Inspect Attention

- `model.encode(...)` / `model.decode(...)` / `model.reconstruct(...)`
- `model.generate(conditions, n_cells=...)`: generate cells for a single condition dictionary or condition table.
- `model.generate_like(target_adata)`: match the exact condition distribution of `target_adata` in a **single batched pass**.
- `model.generate_with_attention(...)`: inspect DiT block attention weights alongside generated cells.

In [ ]:
# 1. Encode to pathway-block latent space (N, L, enc_dim) and reconstruct
latents = model.encode(with_cond=False, latent_pos=1)  # posterior mean mu
decoded = model.decode(latents[:5], cond={"batch": "10x_v2", "cell_type": "CD4T"})
recon = model.reconstruct()
print(f"Latents: {latents.shape} | Decoded: {decoded.shape} | Reconstructed: {recon.shape}")

# 2. Generate synthetic cells for a specific condition
gen_single = model.generate(
    {"batch": "10x_v3", "cell_type": "B"},
    n_cells=15,
    guidance_weight=2.0,
    steps=10,
    solver="euler",
)
print(f"Generated single condition: {gen_single.shape}")

# 3. Single-pass batched generation matching a target AnnData's condition counts
gen_like, gen_z_fm = model.generate_like(
    block_res.adata[:24],
    guidance_weight=2.0,
    steps=10,
    return_latents=True,
)
print(f"generate_like: adata={gen_like.shape}, fm_latents={gen_z_fm.shape}")

# 4. Generate with DiT attention weights for gene-block interpretability
attn_out = model.generate_with_attention(
    {"batch": "10x_v2", "cell_type": "CD4T"},
    n_cells=4,
    steps=5,
    batch_size=4,
)
print(f"Attention-tracked generation: {attn_out['adata'].shape}, steps recorded={len(attn_out['attn_weights'])}")

## 5. Zero-Shot Condition Transfer & Model Persistence (`save` / `load`)

In [ ]:
# Transfer 10x_v2 cells into 10x_v3 while preserving each cell's own cell_type from obs
src_v2 = block_res.adata[block_res.adata.obs["batch"] == "10x_v2"][:10].copy()
transferred = model.transfer(
    src_v2,
    target_cond={"batch": "10x_v3"},
    input_cond="obs",
    inverse_scale=True,
    trim_pad=True,
)
print("Transferred batches:", transferred.obs["batch"].value_counts().to_dict())

# Save full model bundle (config, vocab, metadata, scaler, vae.pt, fm.pt) and reload
with tempfile.TemporaryDirectory() as tmp_dir:
    bundle_dir = Path(tmp_dir) / "save_bundle"
    model.save(bundle_dir)
    print("Saved files:", sorted(p.name for p in bundle_dir.iterdir()))

    reloaded = SAVE.load(bundle_dir, device="auto")
    z_orig = model.encode(src_v2, with_cond=False, latent_pos=1)
    z_reload = reloaded.encode(src_v2, with_cond=False, latent_pos=1)
    max_diff = float(np.max(np.abs(z_orig - z_reload)))
    print(f"Reload parity max |z_orig - z_reload| = {max_diff:.2e}")

## 6. Unified `sc-save` CLI

Every stage of SAVE is available via the `sc-save` CLI (`prepare-datasets`, `build-blocks`, `train`, `generate`, `evaluate`, `bec`, `perturb`, `reproduce`).

In [ ]:
import shutil
import subprocess
import sys

cli = shutil.which("sc-save")
base_cmd = [cli] if cli else [sys.executable, "-m", "sc_save.cli"]
for flag in ("--version", "--help"):
    proc = subprocess.run([*base_cmd, flag], capture_output=True, text=True, check=True)
    print(f"$ sc-save {flag}")
    print(proc.stdout.strip())
    print()